# TabPFN Benchmark Results

Comparison of TabPFN against baseline algorithms on sklearn datasets.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

results_dir = Path("../results")

# NOTE: This notebook must be run from benchmarks/notebooks/ directory,
# or adjust the path below if running from repo root:
# results_dir = Path("benchmarks/results")

clf_path = results_dir / "classification_results.csv"
reg_path = results_dir / "regression_results.csv"

clf_df = pd.read_csv(clf_path) if clf_path.exists() else None
reg_df = pd.read_csv(reg_path) if reg_path.exists() else None

print("Classification results loaded:", clf_df is not None)
print("Regression results loaded:", reg_df is not None)

## Classification Results

In [ ]:
if clf_df is not None:
    metrics = ["accuracy", "f1_macro", "balanced_accuracy", "roc_auc"]
    available = [m for m in metrics if m in clf_df.columns]
    summary = clf_df.groupby("model")[available].mean().sort_values(
        "accuracy", ascending=False,
    )
    display(summary.style.format("{:.4f}").background_gradient(cmap="YlGn", axis=0))

In [ ]:
if clf_df is not None:
    datasets = clf_df["dataset"].unique()
    fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 5))
    if len(datasets) == 1:
        axes = [axes]

    for ax, ds_name in zip(axes, datasets):
        subset = clf_df[clf_df["dataset"] == ds_name].sort_values(
            "accuracy", ascending=True,
        )
        colors = ["#2196F3" if m == "TabPFN" else "#90A4AE"
                  for m in subset["model"]]
        ax.barh(subset["model"], subset["accuracy"], color=colors)
        ax.set_xlabel("Accuracy")
        ax.set_title(f"{ds_name}")
        ax.set_xlim(0, 1)

    plt.suptitle("Classification Accuracy by Dataset", fontsize=14)
    plt.tight_layout()
    plt.savefig(str(results_dir / "classification_accuracy.png"), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
if clf_df is not None:
    time_summary = clf_df.groupby("model")[["fit_time_s", "predict_time_s"]].mean()
    time_summary = time_summary.sort_values("fit_time_s", ascending=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    colors = ["#2196F3" if m == "TabPFN" else "#90A4AE"
              for m in time_summary.index]
    ax1.barh(time_summary.index, time_summary["fit_time_s"], color=colors)
    ax1.set_xlabel("Fit Time (s)")
    ax1.set_title("Average Fit Time")

    ax2.barh(time_summary.index, time_summary["predict_time_s"], color=colors)
    ax2.set_xlabel("Predict Time (s)")
    ax2.set_title("Average Predict Time")

    plt.tight_layout()
    plt.savefig(str(results_dir / "classification_timing.png"), dpi=150, bbox_inches="tight")
    plt.show()

## Regression Results

In [ ]:
if reg_df is not None:
    metrics = ["mse", "rmse", "mae", "r2"]
    available = [m for m in metrics if m in reg_df.columns]
    summary = reg_df.groupby("model")[available].mean().sort_values(
        "r2", ascending=False,
    )
    display(summary.style.format("{:.4f}").background_gradient(cmap="YlGn", axis=0, subset=["r2"]))

In [ ]:
if reg_df is not None:
    datasets = reg_df["dataset"].unique()
    fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 5))
    if len(datasets) == 1:
        axes = [axes]

    for ax, ds_name in zip(axes, datasets):
        subset = reg_df[reg_df["dataset"] == ds_name].sort_values(
            "r2", ascending=True,
        )
        colors = ["#2196F3" if m == "TabPFN" else "#90A4AE"
                  for m in subset["model"]]
        ax.barh(subset["model"], subset["r2"], color=colors)
        ax.set_xlabel("R\u00b2 Score")
        ax.set_title(f"{ds_name}")

    plt.suptitle("Regression R\u00b2 by Dataset", fontsize=14)
    plt.tight_layout()
    plt.savefig(str(results_dir / "regression_r2.png"), dpi=150, bbox_inches="tight")
    plt.show()

## Per-Dataset Detailed Results

In [ ]:
if clf_df is not None:
    print("=== Classification ===")
    for ds_name in clf_df["dataset"].unique():
        print(f"\n--- {ds_name} ---")
        subset = clf_df[clf_df["dataset"] == ds_name].set_index("model")
        cols = [c for c in ["accuracy", "f1_macro", "roc_auc", "fit_time_s"] if c in subset.columns]
        print(subset[cols].sort_values("accuracy", ascending=False).to_string(float_format="%.4f"))

if reg_df is not None:
    print("\n=== Regression ===")
    for ds_name in reg_df["dataset"].unique():
        print(f"\n--- {ds_name} ---")
        subset = reg_df[reg_df["dataset"] == ds_name].set_index("model")
        cols = [c for c in ["r2", "rmse", "mae", "fit_time_s"] if c in subset.columns]
        print(subset[cols].sort_values("r2", ascending=False).to_string(float_format="%.4f"))